# Landscape Metrics: Plots

Reads the tidied/metric CSVs exported by the R pipeline (`scripts/r/`, specifically
`03_landscape_metrics.R`'s fragmentation/connectivity metrics and correlation screen) from
`outputs/tables/`.


In [ ]:
import geopandas as gpd
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import rasterio
import seaborn as sns
from matplotlib.colors import TwoSlopeNorm
from matplotlib.ticker import MaxNLocator
from matplotlib.colors import LinearSegmentedColormap

import config

In [ ]:
sns.set_theme(style="whitegrid")
config.PLOTS_DIR.mkdir(parents=True, exist_ok=True)

SITE_ORDER = [s["site_id"] for s in config.SITES]
SITE_LABELS = {s["site_id"]: s["site_name"] for s in config.SITES}

## Load tables

`landscape_connectivity_metrics_binary_natural_by_site_year_season.csv` is long-format (one row
per site/year/season-or-period/metric) period-composite rows (`baseline_2016_2018` etc.) carry
`year`/`season` as NA, seasonal per-year rows carry `period` as NA. Plots 1-5 below use the
seasonal rows only (`year` not null).
`landscape_metric_correlation_matrix.csv` is long-format pairwise correlations
(`Var1`/`Var2`/`Freq`) from `03_landscape_metrics.R`'s redundancy screen, computed across ALL
those seasonal + period observations pooled together.


In [ ]:
binary_metrics = pd.read_csv(config.TABLES_DIR / "landscape_connectivity_metrics_binary_natural_by_site_year_season.csv")
correlation_matrix = pd.read_csv(config.TABLES_DIR / "landscape_metric_correlation_matrix.csv")

## Plots 1-5: Natural-habitat fragmentation/connectivity metric trends by site

Five of the six headline metrics `03_landscape_metrics.R` computed on the binary natural-habitat
raster (classes 1-3 vs. 4-6), per site/year/season, 2016-2025 wet/dry seasonal composites. Color = site,
line style = season. PLAND is a percentage of the
*classified* (non-NA) area within each site, not of the site's full nominal polygon area. See
`landscape_valid_pixel_coverage_by_site_year_season.csv` for that ratio per site/year/season.


In [ ]:
def plot_landscape_metric_trend(metric: str, ylabel: str, filename: str) -> None:
    """Line plot of one binary natural-habitat landscape metric by site/season, 2016-2025."""
    sub = binary_metrics[(binary_metrics["metric"] == metric) & binary_metrics["year"].notna()].copy()
    sub["site_name"] = sub["site_id"].map(SITE_LABELS)
    sub["year"] = sub["year"].astype(int)

    fig, ax = plt.subplots(figsize=(12, 7))
    sns.lineplot(
        data=sub,
        x="year",
        y="value",
        hue="site_name",
        hue_order=[SITE_LABELS[s] for s in SITE_ORDER],
        style="season",
        markers=True,
        dashes=True,
        ax=ax,
    )
    ax.set_xlabel("Year")
    ax.set_ylabel(ylabel)
    ax.set_title(f"Natural-habitat {metric.upper()} by site")
    ax.legend(title=None, bbox_to_anchor=(1.02, 1), loc="upper left")
    fig.tight_layout()
    fig.savefig(config.PLOTS_DIR / filename, dpi=200, bbox_inches="tight")

### Plot 1: PLAND (% of classified area, natural)

In [ ]:
plot_landscape_metric_trend(
    "pland",
    "PLAND, % of classified area (natural)",
    "landscape_pland_by_site_trend.png",
)

### Plot 2: Patch density (PD)

In [ ]:
plot_landscape_metric_trend(
    "pd",
    "Patch density (patches / 100 ha)",
    "landscape_pd_by_site_trend.png",
)

### Plot 3: Largest patch index (LPI)

In [ ]:
plot_landscape_metric_trend(
    "lpi",
    "Largest patch index (%)",
    "landscape_lpi_by_site_trend.png",
)

### Plot 4: Cohesion

In [ ]:
plot_landscape_metric_trend(
    "cohesion",
    "Cohesion",
    "landscape_cohesion_by_site_trend.png",
)

### Plot 5: Effective mesh size (MESH)

In [ ]:
plot_landscape_metric_trend(
    "mesh",
    "Effective mesh size (ha)",
    "landscape_mesh_by_site_trend.png",
)

## Plot 6: Metric correlation matrix

Pairwise correlation (pooled across every site/year/season/period observation) among the nine
binary natural-habitat metrics (`ai`, `clumpy`, `cohesion`, `ed`, `enn_mn`, `lpi`, `mesh`, `pd`,
`pland`), from `03_landscape_metrics.R`'s redundancy screen -- the entropy pilot
(`landscape_entropy_pilot_by_site_year_season.csv`) is computed separately and not part of this
correlation matrix. A pair with `|r| > 0.85` that tells a similar ecological story should have
one metric dropped from the final six-headline-metric reporting set -- that's a human judgment
call for the report, not something this notebook decides automatically.

In [ ]:
corr_wide = correlation_matrix.pivot(index="Var1", columns="Var2", values="Freq")

fig, ax = plt.subplots(figsize=(8, 7))
sns.heatmap(
    corr_wide,
    annot=True,
    fmt=".2f",
    cmap="RdBu_r",
    center=0,
    vmin=-1,
    vmax=1,
    ax=ax,
    cbar_kws={"label": "r"},
)
ax.set_xlabel("")
ax.set_ylabel("")
ax.set_title("Landscape metric correlation matrix")
fig.tight_layout()
fig.savefig(config.PLOTS_DIR / "landscape_metric_correlation_heatmap.png", dpi=200, bbox_inches="tight")

## Plots 7-8: Fragmentation-pressure context maps

Dynamic World conversion-pressure classification (`pressure_composition_by_period.png`
in `historical_change_plots.ipynb`) uses thresholds that were never calibrated against ground truth
(unlike `DW_HABITAT_THRESHOLDS`'s documented 3-round calibration) -- treat that chart as
preliminary. The two maps below are complementary evidence for the same "where is
pressure/threat coming from" question: `local_edge_density_change_*`/`local_patch_density_change_*`
(`04_moving_window_connectivity.R`) are raw, observed baseline-to-current fragmentation change in
real units, not a classification. `connectivity_plots.ipynb`'s own Plots 1-2 (settlement/road
pressure, Objective 4, built from real GIS vector data rather than an inferred spectral proxy)
round out that same evidence base -- moved there since they read Objective 4 raster outputs, not
this notebook's own `03_landscape_metrics.R` tables.

These are genuinely spatial (unlike Plots 1-6), but -- unlike the patch-graph/vector outputs noted
below, which need real topology exploration in QGIS -- each of these is a single-band continuous
raster that maps cleanly with a static `imshow` + colorbar, so a quick map belongs here too.


In [ ]:
site_boundaries = gpd.GeoDataFrame(
    pd.concat(
        [gpd.read_file(p).to_crs(config.PROJECT_CRS) for p in config.AOI_PATHS.values()],
        ignore_index=True,
    )
)


def plot_raster_map(raster_path, title, filename, cmap, cbar_label, diverging=False, vmin=None, vmax=None):
    """Static map of one single-band continuous raster, with site boundaries for context.

    diverging=True centers the colormap at 0 (TwoSlopeNorm) and, if vmin/vmax aren't given,
    picks a symmetric range from the 98th percentile of |value| -- robust to a few extreme
    per-window outliers rather than letting them wash out the rest of the map.
    """
    with rasterio.open(raster_path) as src:
        arr = src.read(1, masked=True).filled(np.nan)
        left, bottom, right, top = src.bounds

    norm = None
    if diverging:
        if vmax is None:
            vmax = np.nanpercentile(np.abs(arr), 98)
            vmin = -vmax
        norm = TwoSlopeNorm(vcenter=0, vmin=vmin, vmax=vmax)
        vmin = vmax = None  # norm supersedes vmin/vmax in imshow

    fig, ax = plt.subplots(figsize=(9, 8))
    im = ax.imshow(arr, extent=(left, right, bottom, top), origin="upper", cmap=cmap, norm=norm, vmin=vmin, vmax=vmax)
    site_boundaries.boundary.plot(ax=ax, color="#484848", linewidth=0.7)
    ax.set_title(title)
    ax.set_xlabel("Easting (m)", fontsize=10.5)
    ax.set_ylabel("Northing (m)", fontsize=10.5)

    # Full meter notation on both axes
    # offset on the y axis once values get this large; disable it so y matches x's plain notation.
    ax.ticklabel_format(axis="both", style="plain", useOffset=False)
    # Slightly fewer gridlines than the default auto locator (was ~7 per axis at ~2000m spacing).
    ax.xaxis.set_major_locator(MaxNLocator(nbins=5))
    ax.yaxis.set_major_locator(MaxNLocator(nbins=5))
    ax.tick_params(axis="both", labelsize=7.0)
    ax.grid(True, alpha=0.6)  # same grid color as the notebook's whitegrid theme, just lighter

    cbar = fig.colorbar(im, ax=ax, shrink=0.8)
    cbar.set_label(cbar_label, fontsize=10.5)
    cbar.ax.tick_params(labelsize=9.5)

    fig.tight_layout()
    fig.savefig(config.PLOTS_DIR / filename, dpi=200, bbox_inches="tight")


### Plot 7: Local edge-density change, 500m window (Objective 3)

In [ ]:
# positive = more fragmented"
plot_raster_map(
    config.LANDSCAPE_RASTER_DIR / "local_edge_density_change_baseline_to_current_w500m.tif",
    "Local edge-density change, (500m window) baseline to current period",
    "landscape_edge_density_change_w500m_map.png",
    cmap="RdBu_r",
    cbar_label="Edge density change (m/ha)",
    diverging=True,
)

### Plot 8: Local patch-density change, 500m window (Objective 3)

In [ ]:
# positive = more fragmented"
plot_raster_map(
    config.LANDSCAPE_RASTER_DIR / "local_patch_density_change_baseline_to_current_w500m.tif",
    "Local patch-density change, (500m window) baseline to current period",
    "landscape_patch_density_change_w500m_map.png",
    cmap="RdBu_r",
    cbar_label="Patch density change (patches/100ha)",
    diverging=True,
)

## Plot 9: Structural metrics summary (baseline / pre-establishment / current)

Period-level companion to Plots 1-5's seasonal trends (Cohesion, AI, Clumpy, and the entropy pilot are
left out here even though PLAND/PD/LPI/MESH/Cohesion already have their own seasonal-trend plots
above).

Corridor Phase 2's baseline period falls below the 80% valid-pixel-coverage floor
(`VALID_PIXEL_COVERAGE_MIN` in `scripts/r/00_config.R`) -- `03_landscape_metrics.R` still computes
its metrics (doesn't drop the row), but tags every value `below_coverage_threshold = True` and
excludes it from the correlation screen and the metric-change summary CSVs, since low coverage is
associated with artificial patch breaks that inflate PD/ED. This figure plots that value anyway,
as a separate grey "x" marker rather than lowering the QA threshold to let it join the trend
line -- it's genuinely computed, just flagged as not trustworthy enough to treat as equivalent to
the other, fully-covered points. The trend line itself still correctly starts at
"Pre-establishment" for this site.


In [ ]:
def plot_landscape_metrics_summary() -> None:
    """2x3 combined figure of the six binary natural-habitat metrics discussed in the Results
    section, one panel per metric, baseline/pre-establishment/current on the x-axis (period
    composites, not the seasonal annual trend Plots 1-5 use).

    Rows flagged `below_coverage_threshold` (currently just Corridor Phase 2's baseline -- see
    the markdown above) are excluded from the trend line and drawn separately as a grey "x"
    marker at their real value, instead of either dropping them or letting them distort the line.
    A grey dashed segment connects that flagged point to its own site's next chronological
    (trusted) point, purely so the eye can follow the site's trajectory across all three periods
    -- it's deliberately grey/dashed rather than the site's own solid color, so it never reads as
    an equally-trusted segment of the main trend line.
    """
    period_order = ["baseline_2016_2018", "pre_2019_2021", "current_2022_2025"]
    period_labels = {
        "baseline_2016_2018": "Baseline\n2016-2018",
        "pre_2019_2021": "Pre-establishment\n2019-2021",
        "current_2022_2025": "Current\n2022-2025",
    }

    sub = binary_metrics[binary_metrics["year"].isna() & binary_metrics["period"].isin(period_order)].copy()
    sub["site_name"] = sub["site_id"].map(SITE_LABELS)
    sub["period_label"] = pd.Categorical(
        sub["period"].map(period_labels),
        categories=[period_labels[p] for p in period_order],
        ordered=True,
    )
    trusted = sub[~sub["below_coverage_threshold"]]
    flagged = sub[sub["below_coverage_threshold"]]

    panels = [
        ("pland", "PLAND, % of classified area (natural)"),
        ("pd", "Patch density (patches / 100 ha)"),
        ("ed", "Edge density (m / ha)"),
        ("lpi", "Largest patch index (%)"),
        ("mesh", "Effective mesh size (ha)"),
        ("enn_mn", "Mean nearest-neighbor distance (m)"),
    ]
    hue_order = [SITE_LABELS[s] for s in SITE_ORDER]

    fig, axes = plt.subplots(3, 2, figsize=(13, 16))
    for i, (ax, (metric, ylabel)) in enumerate(zip(axes.flat, panels)):
        metric_trusted = trusted[trusted["metric"] == metric]
        sns.lineplot(
            data=metric_trusted,
            x="period_label",
            y="value",
            hue="site_name",
            hue_order=hue_order,
            marker="o",
            markersize=8,
            ax=ax,
            legend=(i == 0),
        )
        metric_flagged = flagged[flagged["metric"] == metric]
        if not metric_flagged.empty:
            for _, frow in metric_flagged.iterrows():
                # Connect to this same site's next chronological trusted point, if it has one --
                # a plain grey/dashed reference line, not a second trend-line segment.
                next_idx = period_order.index(frow["period"]) + 1
                if next_idx < len(period_order):
                    next_point = metric_trusted[
                        (metric_trusted["site_id"] == frow["site_id"])
                        & (metric_trusted["period"] == period_order[next_idx])
                    ]
                    if not next_point.empty:
                        ax.plot(
                            [frow["period_label"], next_point["period_label"].iloc[0]],
                            [frow["value"], next_point["value"].iloc[0]],
                            color="gray",
                            linestyle="--",
                            linewidth=1.5,
                            zorder=4,
                        )
            ax.scatter(
                metric_flagged["period_label"],
                metric_flagged["value"],
                marker="x",
                s=90,
                linewidths=2,
                color="gray",
                zorder=5,
                label="Insufficient coverage (<80%)" if i == 0 else None,
            )
        ax.set_xlabel("")
        ax.set_ylabel(ylabel)
        ax.set_title(metric.upper())

    handles, labels = axes.flat[0].get_legend_handles_labels()
    axes.flat[0].get_legend().remove()
    fig.legend(handles, labels, loc="lower center", ncol=len(hue_order) + 1, bbox_to_anchor=(0.5, 0.0))

    fig.suptitle("Natural-habitat structural metrics by period", fontsize=14)
    fig.tight_layout(rect=(0, 0.04, 1, 0.97))
    fig.savefig(config.PLOTS_DIR / "landscape_structural_metrics_summary.png", dpi=200, bbox_inches="tight")


plot_landscape_metrics_summary()